## Algorytmy generowania i weryfikacji danych losowych
## Podsumowanie projektu 
### Autorzy: Krzysztof Głowacz, Marta Puz, Łukasz Sokołowski

W trakcie trwania projektu znalieziono, zaimplementowano oraz przetestowano 20 generatorów. Poszczególne generatory charakteryzują się innymi cechami: czas działanioa, czy kryptograicznie bezpieczny, jak szybko działa, jaki jest okres generowania, czy jest wrażliwy na zeroload?

### Kladyfikacja zebranych generatorów

1. **Klasyczne PRNG** - Historyczne generatory pseudolosowe 
    - LCG
    - Mersenne Twister
2. **Nowocześniejsze PRNG** - Szybkie generatory nowej generacji 
    - Xoshiro256StarStar 
    - SplitMix64 
    - WELL512a
    - JSF32
    - PCG64 (Permuted Congruential Generator)
3. **Kryptograficzne CSPRNG oparte o szyfrowanie/funkcje skrótu** - Bezpieczne kryptograficznie generatory 
    - ChaCha20 
    - HMAC-DRBG 
    - Mod Exp Feistel (wg autorów)
    - SystemCSPRNG (system-backed: os.urandom, secrets)
4. **Counter-Based RNG** - Generatory bez stanu bazowane na licznikach 
    - Philox 
    - Threefry
5. **Multiple Recursive** - Generatory oparte na rekurencji wielokrotnej 
    - MRG32k3a
6. **Chaotyczne** - Generatory oparte na dynamice chaotycznej 
    - Logistic Map
    - HCCS
7. **TRNG / Fizyczne** - Generatory prawdziwie losowe bazujące na entropii fizycznej
    - trng_mock
8. **QRNG** - Generatory oparte o zjawiska kwantowe
    - ANU  Quantum RNG (fluktuacje próżni)

8. **Inne** - Inne generatory
    - µRNG - unikatowy i minimalistycznya, oparty na nieoczywistej teorii liczb p-adycznych
    - Blum-Blum-Shub - teoretycznie CSPRNG, rzadko stosowan ze względu na wydajność, generuja bit po bicie
    - GAN-based - podejście w oparciu o modele generatywne (brak implementacji)


### Opis implementacji generatorów

Wszystkie implementowane generatory w projekcie muszą implementować **RandomGenerator** definiujący metody:

| Metoda | Argumenty | Zwraca | Opis |
|--------|-----------|--------|------|
| `random_uint64(size)` | `size: int` | `list[int]` | Zwraca listę `size` losowych liczb całkowitych 64-bitowych bez znaku (0 do 2^64-1) |
| `random_bytes(nbytes)` | `nbytes: int` | `bytes` | Zwraca `nbytes` losowych bajtów (wartości 0-255) |
| `random_bits(nbits)` | `nbits: int` | `list[int]` | Zwraca listę `nbits` losowych bitów (wartości 0 lub 1) |
| `random_floats(size)` | `size: int` | `list[float]` | Zwraca listę `size` liczb zmiennoprzecinkowych z przedziału [0, 1) |

**Wymagane atrybuty:**
- `name: str` – Nazwa generatora 

**Korzyści wspólnego interfejsu**:
 - Łatwość testowania - Kod testujący generatory jest niezmienny
 - Większa czytelność 
 - Weryfikacja spójnych właściwości (długość wyjścia, zakres wartości itp.)
 - Każdy generator może zastąpić inny bez zmian w kodzie

### Opis wykonywanych testów

Konfiguracja testów walidacyjnych (ValidationConfig) składa się:
- **n_numbers** - Liczba generowanych liczb całkowitych 64-bit
- **n_bits** - Liczba generowanych bitów
- **approx_entropy_block_size** - Rozmiar bloku dla testu entropii przybliżonej

Metoda **validate_generator** przeprowadza kompleksową walidację generatora, sprawdzając generowanie:
- liczb całkowitych 64-bitowych
- liczb zmiennoprzecinkowych w zakresie [0, 1)
- bitów 0/1

Składa się z:
- podstawowych statystyk: mean, variance, std, min, max dla liczb całkowitych i floatów
- bilansu zer i jedynek
- częstości par bitów (00, 01, 10, 11)
- zestawu testów statystycznych:

| Test | Dane wejściowe | Co sprawdza | Kryterium zaliczenia |
|------|----------------|-------------|----------------------|
| `monobit_test` | Bity | Czy liczba jedynek jest zbliżona do liczby zer | `p_value >= 0.01` |
| `runs_test` | Bity | Czy długość serii bitów jest zgodna z oczekiwaniami losowego ciągu | `p_value >= 0.01` |
| `chi_square_uniform_test` | Liczby 64-bit modulo 256 | Równomierność rozkładu wartości | `p_value >= 0.01` |
| `kolmogorov_smirnov_uniform_test` | Floaty [0,1) | Czy rozkład jest równomierny na [0, 1) | `p_value >= 0.01` |
| `approximate_entropy_test` | Bity | Czy ciąg bitów jest wystarczająco nieprzewidywalny dla bloków długości `m` | `p_value >= 0.01` |

### Wizualizacja generowanych liczb

Wykresy wizualnie pokazują różne aspekty jakości generatora: rozkład, zależności sąsiednie, losowość bitów i sygnałowy obraz szumu. Pozwalają na szybką i pobieżną ocenę jakości wyników. Funkcja pobiera n próbek, na których podstawie tworzy wykresy:

- **Histogram wartości** - Sprawdza, czy rozkład jest równomierny na przedziale [0, 1].
- **Wykres rozrzutu 2D** - Tworzy scatter plot par (x_n, x_{n+1}) dla sąsiednich wartości do wizualnej analizy korelacji.
- **Obraz szumu** - Tworzy macierz wartości 0-255 i wyświetla ją jako obraz w skali szarości. Pozwala ocenić wizualnie, czy bajty wyglądają jak losowy szum.
- **Błądzenie losowe 1D** - Generuje 10 000 bitów i przekształca bity na kroki +1 / -1. Rysuje ścieżkę, która pokazuje dynamikę losowości.